# 4.1a Monitoring the fraud model in production

**Story:** The SafeDrive fraud model is live. A tuned model (v2) is ready, claims are getting more
expensive in 2026, and the operations team wants to know quickly when something breaks.
You will A/B test v2, shadow test it, watch for data drift and set up alerts.

**Task 4.1:** Monitor ML and AI model inference and performance.

| Skill | What you do here | Status |
|---|---|---|
| 4.1.4 | A/B test two production variants, shift traffic | ✅ |
| 4.1.3 | Detect data drift with SageMaker Model Monitor (data quality) | ✅ |
| 4.1.2 | Alarms and EventBridge rules for errors and failed jobs | ✅ |
| 4.1.1 | Drift detection pipeline: drift alarm → EventBridge → retraining pipeline | ✅ |
| 2.3.3 | Shadow variant next to a production variant | ✅ |

💰 This notebook runs 2 x `ml.m5.large` endpoint instances (about USD 0.25 per hour in total) and two
short processing jobs. Run the **Cleanup** section when you finish.

## Key points for the exam

- **Model Monitor** needs **data capture** on the endpoint (or a batch transform input), a **baseline**
  (statistics + constraints) and a **monitoring schedule**. Results: `constraint_violations.json` in S3
  and CloudWatch metrics.
- Four monitor types:

| Monitor | Detects | Baseline from | Extra need |
|---|---|---|---|
| Data quality | Drift in input data (types, missing values, distribution) | Training data | none |
| Model quality | Drop in accuracy, AUC, RMSE ... | Predictions + labels | **Ground truth labels** merged later |
| Bias drift | Change in bias metrics (for example DPPL) | Clarify bias baseline | Ground truth labels |
| Feature attribution drift | Change in SHAP feature ranking | Clarify explainability baseline | none |

- **A/B test** = two **production variants** on one endpoint; traffic split by **variant weight**;
  compare per-variant CloudWatch metrics; shift with `UpdateEndpointWeightsAndCapacities` (no downtime).
- **Shadow variant** gets a *copy* of the requests; its responses are **not returned** to the caller.
  Use it to test latency and errors of a new model with zero user risk.
- **Data drift** = input distribution changes. **Concept drift** = relation between input and label
  changes (needs labels, so use model quality monitoring).
- Job and pipeline failures arrive as **EventBridge** events; endpoint errors and latency are
  **CloudWatch metrics** (namespace `AWS/SageMaker`).

| Choose | When |
|---|---|
| A/B (production variants) | You want to compare business results on real users |
| Shadow variant | You want to check a new model safely before users see it |
| Canary / linear (blue/green guardrails) | You want a safe rollout with automatic rollback on alarms |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, time, datetime
import numpy as np, pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C, features as F
print(U.REGION, U.bucket())

Catch-up: make sure the training data and the baseline fraud model exist (returns at once if they do).

In [ ]:
C.ensure_fraud_model()
state = U.load_state()
MODEL_V1 = state["fraud_model_data"]
MODEL_V2 = state.get("fraud_model_data_v2", MODEL_V1)   # tuned model from 2.2a, or the same artifact
print("v1:", MODEL_V1)
print("v2:", MODEL_V2, "(same artifact as v1)" if MODEL_V2 == MODEL_V1 else "")

Clients, names and two small helpers used in the whole notebook.

In [ ]:
sm, smr = U.client("sagemaker"), U.client("sagemaker-runtime")
cw, events, sns = U.client("cloudwatch"), U.client("events"), U.client("sns")

EP = "safedrive-fraud-ab"
CFG_AB, CFG_SHADOW = "safedrive-fraud-ab-config", "safedrive-fraud-shadow-config"
MON_PREFIX = f"{U.PREFIX}/monitoring"
CAPTURE_URI = U.s3_uri(f"{MON_PREFIX}/datacapture")

def exists(fn, **kwargs):
    try:
        fn(**kwargs); return True
    except ClientError:
        return False

key_of = lambda uri: uri.split("/", 3)[3]     # s3://bucket/key -> key
print(CAPTURE_URI)

## 4.1.4 A/B testing with production variants

- One endpoint, two **production variants** (`v1`, `v2`), each with its own model and instances.
- `InitialVariantWeight` sets the traffic split (here 50/50).
- `DataCaptureConfig` saves requests and responses to S3. Model Monitor reads them later.

Create one SageMaker model per version. Look at the two model names printed.

In [ ]:
for name, data in [("safedrive-fraud-ab-v1", MODEL_V1), ("safedrive-fraud-ab-v2", MODEL_V2)]:
    if not exists(sm.describe_model, ModelName=name):
        sm.create_model(ModelName=name, ExecutionRoleArn=U.role_arn(), Tags=U.TAGS,
                        PrimaryContainer={"Image": state["xgb_image"], "ModelDataUrl": data})
    print("model ready:", name)

Create the endpoint configuration: two variants with equal weight, and data capture at 100%.

In [ ]:
def variant(name, model, weight=1.0):
    return {"VariantName": name, "ModelName": model, "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1, "InitialVariantWeight": weight}

capture = {"EnableCapture": True, "InitialSamplingPercentage": 100, "DestinationS3Uri": CAPTURE_URI,
           "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
           "CaptureContentTypeHeader": {"CsvContentTypes": ["text/csv"]}}

if not exists(sm.describe_endpoint_config, EndpointConfigName=CFG_AB):
    sm.create_endpoint_config(
        EndpointConfigName=CFG_AB, Tags=U.TAGS, DataCaptureConfig=capture,
        ProductionVariants=[variant("v1", "safedrive-fraud-ab-v1"), variant("v2", "safedrive-fraud-ab-v2")])
print([v["VariantName"] for v in sm.describe_endpoint_config(EndpointConfigName=CFG_AB)["ProductionVariants"]])

Create the endpoint. This starts 2 instances (💰 billing starts now).

In [ ]:
if not exists(sm.describe_endpoint, EndpointName=EP):
    sm.create_endpoint(EndpointName=EP, EndpointConfigName=CFG_AB, Tags=U.TAGS)
print(sm.describe_endpoint(EndpointName=EP)["EndpointStatus"])

Wait until the endpoint is `InService` (about 5-8 minutes).

In [ ]:
U.wait_until(lambda: sm.describe_endpoint(EndpointName=EP)["EndpointStatus"],
             ok=("InService",), bad=("Failed",), delay=30, label="endpoint")

Load the test set. The files have no header and the label is the first column.

In [ ]:
test = U.read_csv(key_of(state["test_s3"]), header=None)
y_test, X_test = test.iloc[:, 0].values, test.iloc[:, 1:].values
print(X_test.shape, "fraud rate:", y_test.mean().round(3))

A helper that sends **one claim per request** and returns the score and the variant that answered.

In [ ]:
def score(row, variant=None):
    args = dict(EndpointName=EP, ContentType="text/csv", Body=",".join(str(v) for v in row))
    if variant:
        args["TargetVariant"] = variant          # force a specific variant
    resp = smr.invoke_endpoint(**args)
    return float(resp["Body"].read().decode().strip()), resp["InvokedProductionVariant"]

print(score(X_test[0]))

Send 400 test claims. SageMaker picks the variant by weight. Look at the split: it should be near 50/50.

In [ ]:
N = 400
results = pd.DataFrame([score(r) for r in X_test[:N]], columns=["score", "variant"])
results["label"] = y_test[:N]
print(results["variant"].value_counts())

Compare the variants on **model quality**. We know the labels of the test set, so we can compute AUC
for each variant. In real life the labels (confirmed fraud) arrive days later.

In [ ]:
from sklearn.metrics import roc_auc_score
summary = results.groupby("variant").apply(
    lambda g: pd.Series({"requests": len(g), "auc": roc_auc_score(g["label"], g["score"]),
                         "mean_score": g["score"].mean()}))
summary

Compare the variants on **operational metrics** from CloudWatch. Every endpoint metric has the
dimensions `EndpointName` and `VariantName`. Metrics can take 1-2 minutes to show up; rerun if empty.

In [ ]:
def variant_metric(metric, variant, stat="Sum", minutes=30):
    now = datetime.datetime.now(datetime.timezone.utc)
    points = cw.get_metric_statistics(
        Namespace="AWS/SageMaker", MetricName=metric, Statistics=[stat], Period=60 * minutes,
        StartTime=now - datetime.timedelta(minutes=minutes), EndTime=now,
        Dimensions=[{"Name": "EndpointName", "Value": EP}, {"Name": "VariantName", "Value": variant}])
    return sum(p[stat] for p in points["Datapoints"]) if stat == "Sum" else \
        np.mean([p[stat] for p in points["Datapoints"]] or [np.nan])

rows = [{"variant": v, "invocations": variant_metric("Invocations", v),
         "avg_model_latency_ms": variant_metric("ModelLatency", v, "Average") / 1000,   # metric is in microseconds
         "5xx_errors": variant_metric("Invocation5XXErrors", v)} for v in ["v1", "v2"]]
pd.DataFrame(rows)

Call one variant on purpose with `TargetVariant`. Useful for testing v2 before giving it real traffic.

In [ ]:
print("v1:", score(X_test[1], "v1"))
print("v2:", score(X_test[1], "v2"))

v2 looks fine, so shift traffic to 10% v1 / 90% v2. This changes weights only: no new instances,
no downtime.

In [ ]:
sm.update_endpoint_weights_and_capacities(
    EndpointName=EP,
    DesiredWeightsAndCapacities=[{"VariantName": "v1", "DesiredWeight": 1},
                                 {"VariantName": "v2", "DesiredWeight": 9}])
U.wait_until(lambda: sm.describe_endpoint(EndpointName=EP)["EndpointStatus"],
             ok=("InService",), bad=("Failed",), delay=10, label="endpoint")

Check the new weights and send 100 requests. About 90 should go to v2.

In [ ]:
for v in sm.describe_endpoint(EndpointName=EP)["ProductionVariants"]:
    print(v["VariantName"], "weight", v["CurrentWeight"], "instances", v["CurrentInstanceCount"])
print(pd.Series([score(r)[1] for r in X_test[N:N + 100]]).value_counts())

## 4.1.3 Detect data drift with Model Monitor (data quality)

- Step 1: **baseline job** on training data → `statistics.json` and `constraints.json`.
- Step 2: **monitoring schedule** compares captured endpoint data with the baseline (hourly at most).
- Step 3: read `constraint_violations.json` and the CloudWatch metrics.
- Captured CSV data has no header. Model Monitor matches columns **by position**, with the model
  **output first**, then the input features. So our baseline file is `prediction` + features.

Build the baseline file: 300 training rows, scored by the endpoint (so the first column is the
prediction). Column names use lowercase letters and `_` only, as Model Monitor recommends.

In [ ]:
base = U.read_csv(key_of(state["baseline_s3"])).sample(300, random_state=0)
base.insert(0, "prediction", [score(r)[0] for r in base.values])
base.columns = [c.lower() for c in base.columns]
BASELINE_URI = U.write_df(base, f"{MON_PREFIX}/baseline/baseline_with_prediction.csv")
print(BASELINE_URI, base.shape)
base.iloc[:3, :6]

Look at what data capture wrote to S3 (it can take 1-2 minutes after the first requests).
Each line is one request with `endpointInput` and `endpointOutput`.

In [ ]:
capture_keys = U.list_keys(f"{MON_PREFIX}/datacapture/{EP}/")
print(len(capture_keys), "capture files, for example:", capture_keys[-1] if capture_keys else "none yet")
if capture_keys:
    record = json.loads(U.read_bytes(capture_keys[-1]).decode().splitlines()[0])
    print(json.dumps(record["captureData"], indent=1)[:600])

Create the monitor object (SageMaker Python SDK). It only describes the processing job resources.

In [ ]:
import sagemaker
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator, EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

sm_session = sagemaker.Session(boto_session=U.session, default_bucket=U.bucket())

def new_monitor():
    return DefaultModelMonitor(role=U.role_arn(), instance_count=1, instance_type="ml.m5.xlarge",
                               volume_size_in_gb=20, max_runtime_in_seconds=1800,
                               sagemaker_session=sm_session, tags=U.TAGS)

monitor = new_monitor()
print(monitor.image_uri)

Run the baseline job (a processing job, about 8-10 minutes). It profiles every column of the baseline.

In [ ]:
monitor.suggest_baseline(baseline_dataset=BASELINE_URI, dataset_format=DatasetFormat.csv(header=True),
                         output_s3_uri=U.s3_uri(f"{MON_PREFIX}/baseline/results"), wait=True, logs=False)
print(monitor.latest_baselining_job.describe()["ProcessingJobStatus"])

Look at the baseline **statistics**: type, missing values and distribution of each column.

In [ ]:
stats = pd.json_normalize(monitor.baseline_statistics().body_dict["features"])
stats[["name", "inferred_type", "numerical_statistics.common.num_missing",
       "numerical_statistics.mean", "numerical_statistics.std_dev"]].head(8)

Look at the suggested **constraints**. `monitoring_config` holds the drift threshold
(`comparison_threshold`, default 0.1). You can edit this file to make checks stricter or looser.

In [ ]:
constraints = monitor.suggested_constraints().body_dict
print(json.dumps(constraints["monitoring_config"], indent=1))
pd.json_normalize(constraints["features"]).head(5)

Create the **hourly** schedule. Hourly is the shortest interval. The first run starts at the top of
the next hour (with a delay of up to 20 minutes).

In [ ]:
SCHEDULE, SCHEDULE_NOW = "safedrive-fraud-data-quality", "safedrive-fraud-data-quality-now"
REPORTS_URI = U.s3_uri(f"{MON_PREFIX}/reports")

def schedule_args(name, cron, **extra):
    return dict(monitor_schedule_name=name, schedule_cron_expression=cron, output_s3_uri=REPORTS_URI,
                endpoint_input=EndpointInput(endpoint_name=EP, destination="/opt/ml/processing/input/endpoint"),
                statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
                enable_cloudwatch_metrics=True, **extra)

if not exists(sm.describe_monitoring_schedule, MonitoringScheduleName=SCHEDULE):
    monitor.create_monitoring_schedule(**schedule_args(SCHEDULE, CronExpressionGenerator.hourly()))
print(sm.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE)["MonitoringScheduleStatus"])

Now the drift. `claims_v2` is January 2026: claim amounts are about 40% higher than in the training
year. Build features with the same code as training (`F.build_features`).

In [ ]:
claims_v2 = U.read_parquet(f"{state['raw_prefix']}/claims_v2.parquet")
policies = U.read_csv(f"{state['raw_prefix']}/policies.csv")
X_new = F.build_features(claims_v2, policies)[F.FEATURES]
print(X_new.shape)
pd.DataFrame({"training": base["claim_amount_log"].describe(),
              "claims_v2": X_new["claim_amount_log"].describe()}).round(3)

**Local drift check** (instant). Two common measures per feature:
- **KS statistic**: largest gap between the two cumulative distributions (0 = same, 1 = totally different).
- **PSI** (population stability index): below 0.1 stable, 0.1-0.25 some drift, above 0.25 strong drift.

In [ ]:
from scipy.stats import ks_2samp

def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(np.clip(actual, edges[0], edges[-1]), edges)[0] / len(actual)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

train_feats = U.read_csv(key_of(state["baseline_s3"]))
drift = pd.DataFrame([{"feature": f, "ks": ks_2samp(train_feats[f], X_new[f]).statistic,
                       "psi": psi(train_feats[f].values, X_new[f].values)} for f in F.FEATURES[:15]])
drift.sort_values("ks", ascending=False).round(3).head(8)

Send the drifted claims to the endpoint so that data capture records them (about 1-2 minutes).

In [ ]:
t0 = time.time()
new_scores = [score(r)[0] for r in X_new.values]
print(f"sent {len(new_scores)} requests in {time.time() - t0:.0f}s; "
      f"mean score {np.mean(new_scores):.3f} vs {results['score'].mean():.3f} on the test set")

You do not have to wait for the hourly run. A **one-time schedule** (`NOW`) analyses the last hour of
captured data right away. Wait 2 minutes first so that the capture files are in S3.

In [ ]:
time.sleep(120)
monitor_now = new_monitor()
if not exists(sm.describe_monitoring_schedule, MonitoringScheduleName=SCHEDULE_NOW):
    monitor_now.create_monitoring_schedule(**schedule_args(
        SCHEDULE_NOW, CronExpressionGenerator.now(),
        data_analysis_start_time="-PT1H", data_analysis_end_time="-PT0H"))
print(sm.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NOW)["MonitoringScheduleStatus"])

The monitoring job takes 10-15 minutes. **Continue with section 4.1.2 below** and come back to the
cell "Read the violations" afterwards.

## 4.1.2 Detect anomalies and errors in workflows

- **Endpoint problems** → CloudWatch metrics in `AWS/SageMaker`: `Invocation4XXErrors`,
  `Invocation5XXErrors`, `ModelLatency` (microseconds), `OverheadLatency`, `Invocations`.
- **Failed jobs and pipelines** → EventBridge events from `aws.sagemaker` (status change events).
- **Unknown "normal"** → CloudWatch **anomaly detection** learns a band from history; no fixed threshold.
- Alarm actions go to **SNS** (email, Lambda, chat) or trigger automation.

Create one SNS topic for all alerts. Its policy lets CloudWatch alarms and EventBridge publish to it.

In [ ]:
topic_arn = sns.create_topic(Name="safedrive-ml-alerts",
                             Tags=[{"Key": t["Key"], "Value": t["Value"]} for t in U.TAGS])["TopicArn"]
policy = {"Version": "2012-10-17", "Statement": [
    {"Sid": "account", "Effect": "Allow", "Principal": {"AWS": "*"}, "Action": "SNS:Publish",
     "Resource": topic_arn, "Condition": {"StringEquals": {"AWS:SourceOwner": U.account_id()}}},
    {"Sid": "eventbridge", "Effect": "Allow", "Principal": {"Service": "events.amazonaws.com"},
     "Action": "SNS:Publish", "Resource": topic_arn}]}
sns.set_topic_attributes(TopicArn=topic_arn, AttributeName="Policy", AttributeValue=json.dumps(policy))
print(topic_arn)

Optional: put your email address here to receive the alerts (you must confirm the email from SNS).

In [ ]:
ALERT_EMAIL = ""       # for example "me@example.com"
if ALERT_EMAIL:
    sns.subscribe(TopicArn=topic_arn, Protocol="email", Endpoint=ALERT_EMAIL)
print("subscribed:", ALERT_EMAIL or "nobody (alerts only change the alarm state)")

Alarm 1: any **5XX error** on variant v2 in one minute. Missing data means "no traffic", so it is
treated as not breaching.

In [ ]:
dims_v2 = [{"Name": "EndpointName", "Value": EP}, {"Name": "VariantName", "Value": "v2"}]
cw.put_metric_alarm(
    AlarmName="safedrive-fraud-ab-5xx", Namespace="AWS/SageMaker", MetricName="Invocation5XXErrors",
    Dimensions=dims_v2, Statistic="Sum", Period=60, EvaluationPeriods=1, Threshold=1,
    ComparisonOperator="GreaterThanOrEqualToThreshold", TreatMissingData="notBreaching",
    AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-fraud-ab-5xx")

Alarm 2: **p90 model latency** above 500 ms for 3 minutes in a row. `ModelLatency` is in
**microseconds**, so the threshold is 500000.

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-fraud-ab-latency", Namespace="AWS/SageMaker", MetricName="ModelLatency",
    Dimensions=dims_v2, ExtendedStatistic="p90", Period=60, EvaluationPeriods=3, Threshold=500000,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-fraud-ab-latency")

Alarm 3: **anomaly detection** on the number of invocations. The alarm fires when the metric leaves
the learned band (2 standard deviations wide). Good for "traffic suddenly dropped or spiked".

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-fraud-ab-invocations-anomaly", EvaluationPeriods=2,
    ComparisonOperator="LessThanLowerOrGreaterThanUpperThreshold", ThresholdMetricId="band",
    TreatMissingData="notBreaching", AlarmActions=[topic_arn], Tags=U.TAGS,
    Metrics=[
        {"Id": "m1", "ReturnData": True, "MetricStat": {
            "Metric": {"Namespace": "AWS/SageMaker", "MetricName": "Invocations", "Dimensions": dims_v2},
            "Period": 300, "Stat": "Sum"}},
        {"Id": "band", "Expression": "ANOMALY_DETECTION_BAND(m1, 2)", "Label": "expected range",
         "ReturnData": True}])
print("alarm created: safedrive-fraud-ab-invocations-anomaly")

Cause an inference error: send text instead of numbers. Read the status code. A bad request from the
client counts in `Invocation4XXErrors`; a crash inside the container counts in `Invocation5XXErrors`.

In [ ]:
try:
    smr.invoke_endpoint(EndpointName=EP, ContentType="text/csv", Body="this,is,not,a,claim")
except ClientError as e:
    print(e.response["Error"]["Code"], "| status from container:", e.response.get("OriginalStatusCode"))
    print(e.response["Error"]["Message"][:200])

Test the alarm wiring without waiting for a real incident: force the alarm into `ALARM` state.
If you subscribed your email you get a message. The alarm returns to its real state after the next
evaluation.

In [ ]:
cw.set_alarm_state(AlarmName="safedrive-fraud-ab-5xx", StateValue="ALARM", StateReason="manual test")
alarms = cw.describe_alarms(AlarmNamePrefix="safedrive-fraud-ab")["MetricAlarms"]
pd.DataFrame([{"alarm": a["AlarmName"], "state": a["StateValue"], "reason": a["StateReason"][:60]} for a in alarms])

EventBridge rule: catch **failed** training jobs, processing jobs and pipeline executions.
`$or` matches any of the three status fields.

In [ ]:
failed_pattern = {
    "source": ["aws.sagemaker"],
    "detail-type": ["SageMaker Training Job State Change", "SageMaker Processing Job State Change",
                    "SageMaker Model Building Pipeline Execution Status Change"],
    "detail": {"$or": [{"TrainingJobStatus": ["Failed"]}, {"ProcessingJobStatus": ["Failed"]},
                       {"currentPipelineExecutionStatus": ["Failed"]}]}}
events.put_rule(Name="safedrive-ml-job-failed", EventPattern=json.dumps(failed_pattern),
                State="ENABLED", Description="Failed SageMaker jobs and pipelines", Tags=U.TAGS)
events.put_targets(Rule="safedrive-ml-job-failed", Targets=[{"Id": "sns", "Arn": topic_arn}])
print("rule created: safedrive-ml-job-failed -> SNS")

Test the pattern with a sample event (no real failure needed). `True` means the rule would fire.

In [ ]:
def sample_event(detail_type, detail):
    return json.dumps({"id": "1", "account": U.account_id(), "source": "aws.sagemaker", "region": U.REGION,
                       "time": "2026-01-15T10:00:00Z", "resources": [], "detail-type": detail_type,
                       "detail": detail})

for status in ["Failed", "Completed"]:
    ev = sample_event("SageMaker Training Job State Change", {"TrainingJobStatus": status})
    print(status, "->", events.test_event_pattern(EventPattern=json.dumps(failed_pattern), Event=ev)["Result"])

## 4.1.3 (continued) Read the violations

Wait for the one-time monitoring run to finish. `CompletedWithViolations` means drift or a data
quality problem was found.

In [ ]:
def last_execution(schedule):
    runs = sm.list_monitoring_executions(MonitoringScheduleName=schedule, SortBy="CreationTime",
                                         SortOrder="Descending", MaxResults=1)["MonitoringExecutionSummaries"]
    return runs[0] if runs else {"MonitoringExecutionStatus": "NotStarted"}

U.wait_until(lambda: last_execution(SCHEDULE_NOW)["MonitoringExecutionStatus"],
             ok=("Completed", "CompletedWithViolations"), bad=("Failed", "Stopped"),
             delay=60, timeout=2400, label="monitoring run")

Read `constraint_violations.json`. Each row is one feature that broke a constraint.
`baseline_drift_check` = the distribution moved more than the threshold.
If the table is empty, the drift stayed below the threshold of 0.1 for the mixed traffic of the last hour.

In [ ]:
report_keys = [k for k in U.list_keys(f"{MON_PREFIX}/reports/") if k.endswith("constraint_violations.json")]
print(report_keys[-1:] or "no violations file (the run completed without violations)")
violations = json.loads(U.read_bytes(report_keys[-1]))["violations"] if report_keys else []
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(violations)

If a run fails, the reason is in the execution details (for example "no captured data in the window").

In [ ]:
run = last_execution(SCHEDULE_NOW)
print(run["MonitoringExecutionStatus"], "|", run.get("FailureReason", "no failure"))
print("processing job:", run.get("ProcessingJobArn", "-").split("/")[-1])

Model Monitor also publishes CloudWatch metrics (because `enable_cloudwatch_metrics=True`).
List them: one `feature_baseline_drift_<feature>` metric per feature. They can take a few minutes to appear.

In [ ]:
DRIFT_NS = "aws/sagemaker/Endpoints/data-metrics"
found = cw.list_metrics(Namespace=DRIFT_NS, Dimensions=[{"Name": "Endpoint", "Value": EP}])["Metrics"]
print(len(found), "metrics")
sorted({m["MetricName"] for m in found})[:10]

## 4.1.1 Drift detection pipeline: drift → alarm → retrain

- Model Monitor writes a drift metric → a **CloudWatch alarm** watches it.
- The alarm state change is an **EventBridge** event → the rule starts the **SageMaker pipeline**
  that retrains and registers the model (built in notebook 3.3a).
- Keep a human approval step in the model registry before the new model is deployed.

```
endpoint → data capture (S3) → Model Monitor schedule → CloudWatch metric → alarm
                                                             → EventBridge rule → SageMaker pipeline → registry
```

Step 1: alarm on the drift metric of `claim_amount_log` from the hourly schedule.

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-fraud-drift-claim-amount", Namespace=DRIFT_NS,
    MetricName="feature_baseline_drift_claim_amount_log",
    Dimensions=[{"Name": "Endpoint", "Value": EP}, {"Name": "MonitoringSchedule", "Value": SCHEDULE}],
    Statistic="Maximum", Period=3600, EvaluationPeriods=1, Threshold=0.1,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-fraud-drift-claim-amount")

Step 2: EventBridge rule that fires when this alarm goes to `ALARM`.

In [ ]:
drift_pattern = {"source": ["aws.cloudwatch"], "detail-type": ["CloudWatch Alarm State Change"],
                 "detail": {"alarmName": ["safedrive-fraud-drift-claim-amount"],
                            "state": {"value": ["ALARM"]}}}
events.put_rule(Name="safedrive-drift-retrain", EventPattern=json.dumps(drift_pattern), State="ENABLED",
                Description="Start retraining when drift is detected", Tags=U.TAGS)
print("rule created: safedrive-drift-retrain")

Step 3: the target is the retraining pipeline. EventBridge starts a pipeline execution directly
(no Lambda needed). This cell adds the target only if the pipeline from notebook 3.3a exists.

In [ ]:
pipeline_name = state.get("pipeline_name")
if pipeline_name and exists(sm.describe_pipeline, PipelineName=pipeline_name):
    pipeline_arn = sm.describe_pipeline(PipelineName=pipeline_name)["PipelineArn"]
    events.put_targets(Rule="safedrive-drift-retrain", Targets=[
        {"Id": "retrain", "Arn": pipeline_arn, "RoleArn": U.role_arn(),
         "SageMakerPipelineParameters": {"PipelineParameterList": []}}])
    print("target added:", pipeline_arn)
else:
    print("No pipeline yet (run notebook 3.3a). The rule exists but has no target.")

Other ways to build the same loop:
- Rule on the **monitoring processing job** event and a Lambda that reads `constraint_violations.json`.
- A scheduled pipeline with `ClarifyCheckStep` / `QualityCheckStep` that compares new data with the
  registered baseline and fails (or retrains) on drift.

## 2.3.3 Shadow variant

- An endpoint config can have **one production variant + one shadow variant** (`ShadowProductionVariants`).
- The shadow variant receives a copy of the requests. Callers only get the production response.
- Compare with per-variant CloudWatch metrics (latency, errors) and with data capture.
- The shadow weight compared with the production weight sets how much traffic is copied
  (equal weights = 100%).

Create a new endpoint config: v1 serves users, v2 runs in the shadow. Data capture stays on.

In [ ]:
if not exists(sm.describe_endpoint_config, EndpointConfigName=CFG_SHADOW):
    sm.create_endpoint_config(
        EndpointConfigName=CFG_SHADOW, Tags=U.TAGS, DataCaptureConfig=capture,
        ProductionVariants=[variant("v1", "safedrive-fraud-ab-v1")],
        ShadowProductionVariants=[variant("shadow-v2", "safedrive-fraud-ab-v2")])
cfg = sm.describe_endpoint_config(EndpointConfigName=CFG_SHADOW)
print("production:", [v["VariantName"] for v in cfg["ProductionVariants"]],
      "| shadow:", [v["VariantName"] for v in cfg["ShadowProductionVariants"]])

Apply the config to the running endpoint. SageMaker does a blue/green update: new instances start,
traffic moves, old instances stop (about 8-10 minutes, no downtime).

In [ ]:
sm.update_endpoint(EndpointName=EP, EndpointConfigName=CFG_SHADOW)
U.wait_until(lambda: sm.describe_endpoint(EndpointName=EP)["EndpointStatus"],
             ok=("InService",), bad=("Failed",), delay=30, label="endpoint")

Send 200 requests. Every response comes from `v1`: the caller never sees the shadow variant.

In [ ]:
shadow_run = pd.Series([score(r)[1] for r in X_test[:200]])
print(shadow_run.value_counts())
ep_desc = sm.describe_endpoint(EndpointName=EP)
print("shadow variants:", [v["VariantName"] for v in ep_desc.get("ShadowProductionVariants", [])])

Compare production and shadow in CloudWatch (wait 1-2 minutes, rerun if the shadow row is empty).
Look at latency and errors of `shadow-v2`: if they are fine, promote v2 with an A/B test or a
blue/green deployment.

In [ ]:
rows = [{"variant": v, "invocations": variant_metric("Invocations", v, minutes=15),
         "avg_model_latency_ms": variant_metric("ModelLatency", v, "Average", minutes=15) / 1000,
         "5xx_errors": variant_metric("Invocation5XXErrors", v, minutes=15)} for v in ["v1", "shadow-v2"]]
pd.DataFrame(rows)

📝 **SageMaker inference experiments** manage the same thing for you: `create_inference_experiment`
with `Type="ShadowMode"`, a `ShadowModeConfig` (source variant, shadow variant, sampling percentage)
and a schedule. The console shows a side-by-side dashboard, and you finish the experiment by
promoting or removing the shadow model.

```python
sm.create_inference_experiment(
    Name="safedrive-fraud-shadow-test", Type="ShadowMode", RoleArn=U.role_arn(), EndpointName=EP,
    ModelVariants=[{"ModelName": "safedrive-fraud-ab-v1", "VariantName": "v1", "InfrastructureConfig": {...}},
                   {"ModelName": "safedrive-fraud-ab-v2", "VariantName": "shadow-v2", "InfrastructureConfig": {...}}],
    ShadowModeConfig={"SourceModelVariantName": "v1",
                      "ShadowModelVariants": [{"ShadowModelVariantName": "shadow-v2", "SamplingPercentage": 50}]})
```

## Exam traps

- Model Monitor on a real-time endpoint does nothing without **data capture**. For batch transform
  use `batch_transform_input` instead.
- **Model quality** and **bias drift** monitoring need **ground truth labels**; data quality and
  feature attribution drift do not.
- Shadow variant responses are **not** returned to the client; production variant responses are.
- Changing variant **weights** (`UpdateEndpointWeightsAndCapacities`) needs no new endpoint config;
  changing the **model or instance type** does (`UpdateEndpoint`).
- `ModelLatency` is in **microseconds** and does not include SageMaker overhead (`OverheadLatency`).

## Quiz

**1.** A new fraud model must be tested with real production requests, but customers must only ever
get answers from the current model. What should you use?
A) Two production variants with 50/50 weights  B) A shadow variant  C) A canary deployment  D) A batch transform job

<details><summary>Answer</summary>B. A shadow variant gets a copy of the traffic and its responses are discarded.</details>

**2.** Fraud labels arrive 30 days after a claim. The team wants an alert within a day when incoming
claims look different from the training data. Which monitor fits?
A) Model quality  B) Bias drift  C) Data quality  D) Feature attribution drift

<details><summary>Answer</summary>C. Data quality monitoring compares input distributions with the baseline and needs no labels.</details>

**3.** A monitoring schedule runs every hour but each run fails with "no data found". The endpoint
gets traffic. What is the most likely cause?
A) The baseline has too few rows  B) Data capture is not enabled on the endpoint config
C) The endpoint has two variants  D) CloudWatch metrics are disabled

<details><summary>Answer</summary>B. The schedule analyses captured data in S3; without data capture there is nothing to read.</details>

**4.** In an A/B test variant B wins. You must move all traffic to B with the least effort and no downtime.
A) Delete the endpoint and create a new one  B) Call `UpdateEndpointWeightsAndCapacities` with weight 0 for A
C) Create a new model package  D) Change the data capture sampling to 0

<details><summary>Answer</summary>B. Weights can be changed in place; later remove variant A with a new endpoint config.</details>

**5.** The operations team wants an email when any SageMaker training job fails, with the least custom code.
A) Poll `DescribeTrainingJob` from a Lambda every minute  B) EventBridge rule on training job state change with status Failed → SNS topic
C) CloudWatch alarm on `CPUUtilization`  D) CloudTrail trail with S3 notifications

<details><summary>Answer</summary>B. SageMaker sends job state changes to EventBridge; a rule can send them to SNS.</details>

## Save state

Nothing from this notebook is needed later, so only the S3 locations are saved for reference.

In [ ]:
U.save_state(monitoring_baseline_s3=BASELINE_URI, monitoring_reports_s3=REPORTS_URI)
print("saved")

## Cleanup

Everything created here is deleted (the endpoint is the costly part). The models in S3, the training
data and the files under `safedrive/monitoring/` are kept. Run every cell, in order.

### Before you delete: what later notebooks use

**Nothing from this notebook is needed later.** It is safe to delete everything it created.

Delete the two monitoring schedules first (an endpoint with a schedule cannot be deleted).

In [ ]:
for name in ["safedrive-fraud-data-quality", "safedrive-fraud-data-quality-now"]:
    U.safe(sm.delete_monitoring_schedule, MonitoringScheduleName=name)
time.sleep(30)

Delete the job definitions that the schedules used.

In [ ]:
for jd in sm.list_data_quality_job_definitions(EndpointName="safedrive-fraud-ab")["JobDefinitionSummaries"]:
    U.safe(sm.delete_data_quality_job_definition, JobDefinitionName=jd["MonitoringJobDefinitionName"])

Delete the endpoint (stops the instance cost).

In [ ]:
U.safe(sm.delete_endpoint, EndpointName="safedrive-fraud-ab")

Delete the two endpoint configs.

In [ ]:
for name in ["safedrive-fraud-ab-config", "safedrive-fraud-shadow-config"]:
    U.safe(sm.delete_endpoint_config, EndpointConfigName=name)

Delete the two SageMaker models (the artifacts in S3 stay).

In [ ]:
for name in ["safedrive-fraud-ab-v1", "safedrive-fraud-ab-v2"]:
    U.safe(sm.delete_model, ModelName=name)

Delete the CloudWatch alarms.

In [ ]:
U.safe(cw.delete_alarms, AlarmNames=["safedrive-fraud-ab-5xx", "safedrive-fraud-ab-latency",
                                     "safedrive-fraud-ab-invocations-anomaly",
                                     "safedrive-fraud-drift-claim-amount"])

Delete the EventBridge rules (targets must be removed first).

In [ ]:
for rule, ids in [("safedrive-ml-job-failed", ["sns"]), ("safedrive-drift-retrain", ["retrain"])]:
    U.safe(events.remove_targets, Rule=rule, Ids=ids)
    U.safe(events.delete_rule, Name=rule)

Delete the SNS topic.

In [ ]:
U.safe(sns.delete_topic, TopicArn=f"arn:aws:sns:{U.REGION}:{U.account_id()}:safedrive-ml-alerts")

Check that no endpoint from this notebook is left.

In [ ]:
left = sm.list_endpoints(NameContains="safedrive-fraud-ab")["Endpoints"]
print([(e["EndpointName"], e["EndpointStatus"]) for e in left] or "no endpoint left")